[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Placeholders and Identifiers


## What you will be able to do

Put a value into a query without building the query around it, and say why that is a different thing
from formatting a string. Pass one value without the tuple mistake everybody makes once. Filter on a
list, which needs `= ANY` rather than `IN`, and know what an empty list does. Write many rows with
`executemany`, and get their generated ids back. Put a table or column name into a query, which no
placeholder can carry, using `sql.Identifier` rather than an f-string. And read asyncpg's `$1`,
which is the same idea with different punctuation.


## The idea

### The problem

Every query in this guide so far has had its values written into the SQL, and that has been fine
because the values came from the cell above. The moment one comes from anywhere else, building the
query around it is how a name with an apostrophe in it takes the server down, and how somebody who
sends you a clever name reads your whole database.

The **Why Peewee** notebook opens on the same apostrophe, and the fix there was `?` and a tuple. Here
it is `%s` and a tuple, and the reason it works is worth understanding rather than memorizing,
because psycopg 3 does it differently from the library most examples on the internet were written
for.

### What a placeholder is

Not string formatting. `%s` is not filled in by Python and it is not filled in by psycopg either. The
query travels to the server with the placeholder still in it, the values travel separately, and the
server puts them together after it has finished deciding what the statement means.

That is why a value cannot change the shape of a statement. By the time the value arrives, the
statement has already been parsed, and there is no way for an apostrophe or a semicolon in it to
become syntax.

### Why it works that way

PostgreSQL's protocol has two ways to run a statement: send the text and run it, or send the text to
be prepared, then send values to fill it. psycopg 3 uses the second, which is why the error you get
for a misplaced placeholder names `$1` rather than your value: the server never saw your value at
that point, it saw a parameter marker it could not make sense of where it wanted a name.

psycopg 2 did the opposite, escaping the value in Python and sending one finished string. Code and
answers written for it therefore describe different errors, which is worth knowing when searching.

### Where this shows up

Every query that takes an argument. The list case in particular catches people, because `IN` is what
SQL uses and a Python list is not what `IN` wants.

### What this notebook covers

`%s`, the tuple around a single value, and named placeholders. The list, with `= ANY` and the empty
case. `executemany` and getting ids back. The things no placeholder can carry, and `psycopg.sql` for
building those safely. asyncpg's `$1`. Then the four failures, one of which is silent.

psycopg 3.3 can also take a Python template string as a query, which needs Python 3.14. Colab runs
3.12 at the time of writing, so nothing here uses them.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg

name = "O'Brien"                                # an apostrophe, which ends a SQL string early

with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute(f"SELECT '{name}' AS who")             # the query built by hand
    except psycopg.errors.SyntaxError as error:
        print("built by hand:  ", error)
    conn.rollback()

    print("with a placeholder:", conn.execute("SELECT %s AS who", (name,)).fetchone())
```

```
built by hand:   unterminated quoted string at or near "' AS who"
LINE 1: SELECT 'O'Brien' AS who
                       ^
with a placeholder: ("O'Brien",)
```

The server even draws you a picture: the caret is under the quote that ended the string early. The
second line never had the problem, because the apostrophe was in a value rather than in the
statement, and a value cannot be syntax.


## Setup

Nine imports, both drivers, the server, and a small table to write into.

- `psycopg` and `asyncpg` are the drivers, `errors` is the exception classes, and `sql` is the module
  for building a statement out of names safely
- `subprocess`, `sys`, `os`, `getpass` and `time` stand the server up, which **A Server of Your Own**
  takes apart
- `version` and `PackageNotFoundError` install the drivers where they are missing

`tags` has a `name` that is unique and a `weight`, which is enough to write to, and `build_tags`
empties it so each example starts the same way.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors, sql

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def build_tags():
    """A small table with a unique column, for the writing half of this notebook."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS tags")
        conn.execute("CREATE TABLE tags (id serial PRIMARY KEY, name text UNIQUE, weight int)")


def tags():
    """Everything in tags, as a list of pairs."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT name, weight FROM tags ORDER BY id").fetchall()


print("server:", start_server())
print(report())
build_tags()
print("tags is ready:", tags())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
tags is ready: []


## Worked examples

### The value never becomes part of the statement

Here is the proof, rather than the assertion. A value that is entirely made of SQL:


In [2]:
dangerous = "'; DROP TABLE tags; --"

with psycopg.connect("dbname=guide") as conn:
    got = conn.execute("SELECT %s AS who", (dangerous,)).fetchone()

print("what came back:", got)
print("tags still exists:", tags() is not None)


what came back: ("'; DROP TABLE tags; --",)
tags still exists: True


It came back as a string, because that is what it is. Nothing in it was ever read as SQL, and the
table it asks to drop is still there.

### One value, and the comma everybody forgets

The second argument is a sequence, even when there is one value in it:


In [3]:
with psycopg.connect("dbname=guide") as conn:
    print("a one-item tuple:", conn.execute("SELECT %s", (7,)).fetchone())
    print("a list works too:", conn.execute("SELECT %s", [7]).fetchone())

    try:
        conn.execute("SELECT %s", 7)                                # not a sequence
    except TypeError as error:
        print("a bare value:   ", type(error).__name__ + ":", error)


a one-item tuple: (7,)
a list works too: (7,)
a bare value:    TypeError: object of type 'int' has no len()


`(7)` is the number seven in parentheses and `(7,)` is a tuple, which is a Python fact that becomes a
database bug here. A list is easier to read when the values come from a variable anyway.

Named placeholders take a dictionary, which is worth it once a query has several values and somebody
has to keep their order straight:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    print(conn.execute("SELECT %(kind)s AS kind, %(least)s AS least",
                       {"kind": "purchase", "least": 3}).fetchone())


('purchase', 3)


A literal percent sign in a query with values has to be doubled, because `%` is how psycopg finds
the placeholders:


In [5]:
with psycopg.connect("dbname=guide") as conn:
    print("with values, %% is a percent:", conn.execute("SELECT 'a%%b' AS s, %s", (1,)).fetchone())
    print("with no values, % is fine:   ", conn.execute("SELECT 'a%b' AS s").fetchone())


with values, %% is a percent: ('a%b', 1)
with no values, % is fine:    ('a%b',)


### A list is not an IN list

This is the one that sends people to the internet. `IN` wants a parenthesized list of values, and a
Python list is one value as far as the protocol is concerned:


In [6]:
wanted = [1, 2, 3]

with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT id FROM events WHERE id IN %s", (wanted,))
    except errors.SyntaxError as error:
        print("IN %s ->", str(error).splitlines()[0])
    conn.rollback()

    found = conn.execute("SELECT id FROM events WHERE id = ANY(%s) ORDER BY id", (wanted,)).fetchall()
    print("= ANY(%s) ->", found)


IN %s -> syntax error at or near "$1"
= ANY(%s) -> [(1,), (2,), (3,)]


`= ANY` takes an array, and a Python list is exactly what psycopg turns into one. Nothing is
formatted, the list goes over as a value, and the query text never changes however many items there
are, which is also why it can be prepared once and reused.

The empty list is the case to know about, because it is not an error:


In [7]:
with psycopg.connect("dbname=guide") as conn:
    print("= ANY with []:", conn.execute(
        "SELECT count(*) FROM events WHERE id = ANY(%s)", ([],)).fetchone())
    print("everything:   ", conn.execute("SELECT count(*) FROM events").fetchone())


= ANY with []: (0,)
everything:    (5000,)


Zero rows, raising nothing, which is right: nothing is in an empty list. It is in the Common errors
below because it is usually a bug on the Python side, where a filter came back empty and the code
meant "no filter" rather than "match nothing".

### Many rows at once

`executemany` runs the same statement for each set of values:


In [8]:
build_tags()

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO tags (name, weight) VALUES (%s, %s)",
                    [("sea", 3), ("stone", 1), ("tide", 2)])
    print("rowcount:", cur.rowcount)

print("rows:", tags())


rowcount: 3
rows: [('sea', 3), ('stone', 1), ('tide', 2)]


In psycopg 3 that is not a loop of round trips: `executemany` sends the whole batch through pipeline
mode internally, which is what **Pipeline Mode** is about and what **COPY** is later measured
against.

Generated ids come back if you ask for them, and asking changes how the results are read:


In [9]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO tags (name, weight) VALUES (%s, %s) RETURNING id",
                    [("salt", 4), ("wind", 5)], returning=True)

    ids = []
    while True:
        ids.append(cur.fetchone()[0])
        if not cur.nextset():                                       # one result set per row
            break

print("the ids the server gave:", ids)


the ids the server gave: [4, 5]


`nextset` is there because each set of values produced its own result. That loop is the price of
getting the ids, and **COPY** is the answer when there are enough rows that you would rather not.

### What a placeholder cannot carry

A placeholder is a value, and a table name is not a value. Trying anyway is the first of the Common
errors. `psycopg.sql` is how you build the statement instead:


In [10]:
with psycopg.connect("dbname=guide") as conn:
    query = sql.SQL("SELECT count(*) FROM {}").format(sql.Identifier("tags"))
    print("the statement:", query.as_string(conn))
    print("it runs:      ", conn.execute(query).fetchone())


the statement: SELECT count(*) FROM "tags"
it runs:       (5,)


`Identifier` quotes the name the way PostgreSQL quotes names, which is not the way values are quoted
and is the reason this needs its own tool:


In [11]:
with psycopg.connect("dbname=guide") as conn:
    for name in ("tags", "a name with spaces", 'a "quoted" name'):
        print(f"  {name!r:<22} -> {sql.Identifier(name).as_string(conn)}")


  'tags'                 -> "tags"
  'a name with spaces'   -> "a name with spaces"
  'a "quoted" name'      -> "a ""quoted"" name"


The embedded quote is doubled, which is what makes this safe rather than merely convenient. A column
list is the same idea joined together:


In [12]:
columns = ["name", "weight"]

with psycopg.connect("dbname=guide") as conn:
    query = sql.SQL("SELECT {} FROM {} ORDER BY {}").format(
        sql.SQL(", ").join(sql.Identifier(column) for column in columns),
        sql.Identifier("tags"),
        sql.Identifier("weight"))
    print(query.as_string(conn))
    print(conn.execute(query).fetchall())


SELECT "name", "weight" FROM "tags" ORDER BY "weight"
[('stone', 1), ('tide', 2), ('sea', 3), ('salt', 4), ('wind', 5)]


Two more pieces round it out. `sql.Literal` puts a value into the statement text, quoted, for the
rare case where it cannot be a parameter. `sql.Placeholder` puts a `%s` into a statement you are
composing, so the values still travel separately:


In [13]:
with psycopg.connect("dbname=guide") as conn:
    print("Literal:    ", sql.SQL("WHERE name = {}").format(sql.Literal("O'Brien")).as_string(conn))
    print("Placeholder:", sql.SQL("WHERE name = {}").format(sql.Placeholder()).as_string(conn))

    composed = sql.SQL("SELECT weight FROM {} WHERE name = {}").format(
        sql.Identifier("tags"), sql.Placeholder())
    print("composed:   ", conn.execute(composed, ("sea",)).fetchone())


Literal:     WHERE name = 'O''Brien'
Placeholder: WHERE name = %s
composed:    (3,)


`Placeholder` is the one to reach for. `Literal` is there for statements that cannot take parameters
at all, and using it on anything a person typed puts you back where this notebook started.

### asyncpg, and the other punctuation

asyncpg numbers its placeholders instead of repeating one marker:


In [14]:
conn = await asyncpg.connect(database="guide")

print("$1:      ", await conn.fetchval("SELECT $1::text", "O'Brien"))
print("$1 and $2:", await conn.fetch(
    "SELECT id FROM events WHERE kind = $1 AND id < $2 ORDER BY id", "purchase", 10))
print("a list:  ", await conn.fetch("SELECT id FROM events WHERE id = ANY($1::int[]) ORDER BY id",
                                    [1, 2, 3]))


$1:       O'Brien
$1 and $2: [<Record id=2>, <Record id=5>, <Record id=8>]
a list:   [<Record id=1>, <Record id=2>, <Record id=3>]


Values are positional arguments rather than a sequence, so the tuple mistake cannot happen. The
numbering means a value used twice is written once and referred to twice, which `%s` cannot do
without passing it twice.

A psycopg query pasted over unchanged does not work, and says so clearly, which is the first thing
**asyncpg** covers:


In [15]:
try:
    await conn.fetchval("SELECT %s::text", "O'Brien")
except asyncpg.exceptions.PostgresSyntaxError as error:
    print(type(error).__module__ + "." + type(error).__name__ + ":", error)
await conn.close()


asyncpg.exceptions.PostgresSyntaxError: syntax error at or near "%"


psycopg has a cursor that speaks the same `$1` style, `RawCursor`, which exists for code moving
between the two and for statements that use a value more than once.

### When to reach for which

| What you are putting in | How |
|---|---|
| a value | `%s` and a sequence, or `$1` in asyncpg |
| several values, named | `%(name)s` and a dictionary |
| a list to match against | `= ANY(%s)`, never `IN %s` |
| the same values many times | `cur.executemany(...)` |
| the same values many times, wanting ids | `cur.executemany(..., returning=True)` and `nextset` |
| a table or column name | `sql.SQL(...).format(sql.Identifier(name))` |
| a value inside a composed statement | `sql.Placeholder()`, and pass it as a parameter |
| a value that has to be in the text | `sql.Literal(value)`, and only when it must |

The default is `%s` and a sequence. Reach for `psycopg.sql` only when a name is coming from your own
code, and never build an identifier from anything a person typed without checking it against a list
you control first.

### A search that takes arguments, finished

Everything above, as the function this notebook was for: the column to sort by is a name, the filters
are values, and neither is formatted into the statement by hand.


In [16]:
SORTABLE = {"id", "kind"}                                           # a list you control


def search(kinds, oldest=None, order_by="id", limit=5):
    """Events of any of these kinds, optionally from an id onward, sorted by a column you name."""
    if order_by not in SORTABLE:
        raise ValueError(f"cannot sort by {order_by!r}")

    query = sql.SQL("SELECT id, kind FROM events "
                    "WHERE kind = ANY({kinds}) AND id >= {oldest} "
                    "ORDER BY {order_by} LIMIT {limit}").format(
        kinds=sql.Placeholder("kinds"), oldest=sql.Placeholder("oldest"),
        order_by=sql.Identifier(order_by), limit=sql.Placeholder("limit"))

    with psycopg.connect("dbname=guide") as conn:
        return conn.execute(query, {"kinds": kinds, "oldest": oldest or 0, "limit": limit}).fetchall()


print("two kinds: ", search(["purchase", "click"], limit=4))
print("none:      ", search([], limit=4), "<- an empty list matches nothing")
try:
    search(["click"], order_by="; DROP TABLE tags")
except ValueError as error:
    print("a column name that is not one:", error)


two kinds:  [(2, 'purchase'), (3, 'click'), (5, 'purchase'), (6, 'click')]
none:       [] <- an empty list matches nothing
a column name that is not one: cannot sort by '; DROP TABLE tags'


The kinds, the id and the limit are parameters, so they can be anything. The column name is checked
against a set this module owns before it is ever composed, which is the rule for identifiers: not
"escape it carefully" but "choose it from a list you wrote".

### Where each part came from

| In the search | What it relies on | The section that showed it |
|---|---|---|
| `kind = ANY({kinds})` | a list as one array value | A list is not an IN list |
| `sql.Placeholder("kinds")` | a value in a composed statement | What a placeholder cannot carry |
| `sql.Identifier(order_by)` | a name quoted as a name | What a placeholder cannot carry |
| `SORTABLE` checked first | an identifier chosen, not escaped | What a placeholder cannot carry |
| the dictionary of values | named placeholders | One value, and the comma |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/04-placeholders-and-identifiers-solutions.ipynb).

**1.** Select a string containing an apostrophe and a semicolon as a value, and show it comes back
unchanged.


In [17]:
# your code here


**2.** Count the events whose id is in a list of five ids, using the operator that takes a list.


In [18]:
# your code here


**3.** Write three tags with one call, then write two more and collect the ids the server generated.


In [19]:
# your code here


**4.** Build a query that counts the rows of a table whose name is in a variable, and print the
statement before running it.


In [20]:
# your code here


**5.** Show what `sql.Identifier` does to a name containing a double quote.


In [21]:
# your code here


**6.** Ask asyncpg for the same row twice, once with `$1` and once with a value used in two places.


In [22]:
# your code here


## Common errors

### psycopg.errors.SyntaxError: syntax error at or near "$1"


In [23]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("SELECT count(*) FROM %s", ("events",))


SyntaxError: syntax error at or near "$1"
LINE 1: SELECT count(*) FROM $1
                             ^

The message names `$1` rather than `events`, which is the clearest possible evidence for what a
placeholder is: the server was handed a statement with a parameter marker where a table name belongs,
and it never saw the word `events` at all, because values arrive after parsing.

This is also where psycopg 3 and psycopg 2 differ visibly. psycopg 2 built the string in Python, so
the same mistake there produced a complaint about `'events'`, quoted as a value. An answer on the
internet describing that message is describing the older library.

`sql.Identifier` is the fix, and the same error appears for `IN %s`, for the same reason:


In [24]:
with psycopg.connect("dbname=guide") as conn:
    print(conn.execute(
        sql.SQL("SELECT count(*) FROM {}").format(sql.Identifier("events"))).fetchone())

    try:
        conn.execute("SELECT id FROM events WHERE id IN %s", ([1, 2],))
    except errors.SyntaxError as error:
        print("IN %s says the same thing:", str(error).splitlines()[0])


(5000,)
IN %s says the same thing: syntax error at or near "$1"


### psycopg.ProgrammingError: the query has 2 placeholders but 3 parameters were passed


In [25]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("SELECT %s, %s", (1, 2, 3))


ProgrammingError: the query has 2 placeholders but 3 parameters were passed

Counted by psycopg before anything is sent, which is why this one is a `ProgrammingError` rather than
a message from the server. Too few gives the same error with the numbers the other way round.

It usually means a query was edited and its values were not. Named placeholders make that harder to
get wrong, because the names have to match rather than the count:


In [26]:
with psycopg.connect("dbname=guide") as conn:
    print(conn.execute("SELECT %(a)s, %(b)s", {"a": 1, "b": 2, "unused": 3}).fetchone())
    try:
        conn.execute("SELECT %(a)s, %(b)s", {"a": 1})
    except (errors.ProgrammingError, KeyError) as error:
        print("a missing name:", type(error).__name__ + ":", error)


(1, 2)
a missing name: ProgrammingError: query parameter missing: b


### psycopg.errors.SyntaxError: unterminated quoted string at or near "' AS who"


In [27]:
name = "O'Brien"

with psycopg.connect("dbname=guide") as conn:
    conn.execute(f"SELECT '{name}' AS who")


SyntaxError: unterminated quoted string at or near "' AS who"
LINE 1: SELECT 'O'Brien' AS who
                       ^

The f-string from the first look, as the error it really is. The apostrophe closed the string, and
everything after it was read as SQL.

A name is the polite version of this. The same line with a value somebody else chose is how a table
gets dropped, and no amount of care with quoting fixes it, because the problem is that the value is
in the statement at all:


In [28]:
with psycopg.connect("dbname=guide") as conn:
    print("as a value, it is just a name:", conn.execute("SELECT %s AS who", (name,)).fetchone())
    print("and so is this:               ",
          conn.execute("SELECT %s AS who", ("'; DROP TABLE tags; --",)).fetchone())
print("tags is still here:", len(tags()), "rows")


as a value, it is just a name: ("O'Brien",)
and so is this:                ("'; DROP TABLE tags; --",)
tags is still here: 5 rows


### No error, and no rows at all: = ANY with an empty list


In [29]:
wanted = []                                                         # a filter that came back empty

with psycopg.connect("dbname=guide") as conn:
    found = conn.execute("SELECT count(*) FROM events WHERE id = ANY(%s)", (wanted,)).fetchone()[0]
    total = conn.execute("SELECT count(*) FROM events").fetchone()[0]

print("rows matching an empty list:", found)
print("rows in the table:         ", total)


rows matching an empty list: 0
rows in the table:          5000


The database did exactly what it was asked. The bug is in Python, where an empty list usually means
"no filter was given" rather than "match nothing", and the two are opposite instructions.

Decide which one you mean before the query is built:


In [30]:
def count(kinds=None):
    """Events of these kinds, or all of them when no kinds are given."""
    with psycopg.connect("dbname=guide") as conn:
        if not kinds:                                               # None or empty means no filter
            return conn.execute("SELECT count(*) FROM events").fetchone()[0]
        return conn.execute("SELECT count(*) FROM events WHERE kind = ANY(%s)", (kinds,)).fetchone()[0]


print("no filter:     ", count())
print("an empty list: ", count([]))
print("one kind:      ", count(["purchase"]))


no filter:      5000
an empty list:  5000
one kind:       1667


## Recap

- `%s` is not string formatting. The statement and the values travel separately, and the server puts
  them together after parsing, so a value cannot become syntax.
- The values are a sequence, so one value needs `(value,)` or `[value]`. Named placeholders take a
  dictionary and are easier to keep right as a query grows.
- A literal `%` must be doubled in a query that has values.
- A list needs `= ANY(%s)`, not `IN %s`. An empty list matches nothing and raises nothing, which is
  usually a bug in the code that built it.
- `executemany` writes many rows, and `returning=True` with `nextset` is how the generated ids come
  back.
- A placeholder cannot carry a table or column name. `sql.SQL(...).format(sql.Identifier(name))`
  builds those, quoting them the way names are quoted, and the safe pattern is to choose the name
  from a list you control rather than to escape whatever arrives.
- asyncpg numbers its placeholders, `$1` and `$2`, takes values as positional arguments, and rejects
  `%s` outright.


## What is next

The **Types and Adaptation** notebook is what happens to a value on its way across: every one of them
becomes bytes, most Python types have a rule for that already, and the three that do not are the
ones you have to write a rule for yourself.


---

&#8592; **Previous:** [Transactions and Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/03-transactions-and-errors.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
